<a href="https://colab.research.google.com/github/usman3721/AI-Reseach-MLC-Community/blob/main/mini_maft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-MAFT vs mini-LAFT on XLM-R (Yoruba target, SIB-200 evaluation)

**Question:** with a fixed adaptation budget, is spreading the text over 5 languages (MAFT-style)
about as good for Yoruba as spending all of it on Yoruba (LAFT-style)?

**Languages:** Yoruba, Igbo, Hausa, Swahili, English.
**Text source:** FineWeb2 (ODC-By 1.0) for yor/ibo/hau/swh; FineWeb (English).
**Eval:** SIB-200 topic classification (`Davlan/sib200`), 7 classes.

Four systems are compared: `xlm-roberta-base`, mini-LAFT, mini-MAFT, official AfroXLMR-base.

**Things to verify before trusting results** (marked `CHECK` in the code):
1. English source name (`HuggingFaceFW/fineweb`, `sample-10BT`) and that `Davlan/afro-xlmr-base` loads.
2. Your own "grid" of shot counts: I assumed `SHOTS` below; edit it to match your plan.
3. Yoruba in FineWeb2 is small (~67k docs, ~97 MB), so keep `N_BLOCKS` modest.

In [5]:
!pip install -q transformers datasets accelerate scikit-learn matplotlib pandas

In [6]:
import os, json, random, itertools, gc
import numpy as np, pandas as pd, torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForMaskedLM, AutoModelForSequenceClassification,
                          DataCollatorForLanguageModeling, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)
from sklearn.metrics import f1_score, accuracy_score

print("GPU:", torch.cuda.is_available())

GPU: True


In [7]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/yoruba_project/mini_maft_run'
os.makedirs(WORK, exist_ok=True)
LAFT_DIR, MAFT_DIR = f"{WORK}/mini_laft", f"{WORK}/mini_maft"

Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results



KeyboardInterrupt: 

In [ ]:
# ---------------- CONFIG ----------------
BASE_MODEL = "xlm-roberta-base"
OFFICIAL_AFRO = "Davlan/afro-xlmr-base"

LANGS = ["yor_Latn", "ibo_Latn", "hau_Latn", "swh_Latn", "eng_Latn"]
TARGET = "yor_Latn"

BLOCK = 256                  # tokens per training block
N_BLOCKS = 5_000            # TOTAL blocks per adapted model (~5M tokens). Raise if you have compute.
EVAL_BLOCKS = 200            # held-out blocks per language for MLM loss
MLM_PROB = 0.15
LR = 5e-5                    # paper setting
BATCH = 10                   # paper setting (per device)
EPOCHS = 1                   # one pass over the budget
SEED = 42

SHOTS = [1, 5, 10, 25, 50, 100]   # labelled Yoruba examples per class; ASSUMPTION, edit to your grid
SEEDS = [1, 2, 3]
CLS_MAX_LEN = 128

def fw2(lang):
    """(dataset id, config name) for streaming raw text."""
    if lang == "eng_Latn":
        return "HuggingFaceFW/fineweb", "sample-10BT"      # CHECK: English source
    return "HuggingFaceFW/fineweb-2", lang

## 1. Build the adaptation data
Each language is streamed, tokenised, concatenated and cut into fixed-length blocks.
Yoruba is built once with `N_BLOCKS` blocks. The MAFT mix uses the **first** `N_BLOCKS/5`
Yoruba blocks, so the MAFT Yoruba text is a strict subset of the LAFT text (clean comparison).

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)

def build_blocks(lang, n_blocks, block=BLOCK):
    ds_id, name = fw2(lang)
    stream = load_dataset(ds_id, name=name, split="train", streaming=True)
    inner = block - 2
    buf, out = [], []
    for ex in stream:
        ids = tok(ex["text"], add_special_tokens=False, truncation=False)["input_ids"]
        buf.extend(ids + [tok.sep_token_id])
        while len(buf) >= inner and len(out) < n_blocks:
            chunk, buf = buf[:inner], buf[inner:]
            out.append([tok.cls_token_id] + chunk + [tok.sep_token_id])
        if len(out) >= n_blocks:
            break
    if len(out) < n_blocks:
        print(f"WARNING {lang}: only {len(out)} blocks available (wanted {n_blocks})")
    return out

per_lang = N_BLOCKS // len(LANGS)
need = {l: per_lang + EVAL_BLOCKS for l in LANGS}
need[TARGET] = N_BLOCKS + EVAL_BLOCKS            # Yoruba also feeds the LAFT run

RAW = f"{WORK}/raw_blocks.json"
if os.path.exists(RAW):
    raw = json.load(open(RAW))
else:
    raw = {l: build_blocks(l, need[l]) for l in LANGS}
    json.dump(raw, open(RAW, "w"))
for l in LANGS:
    print(l, len(raw[l]), "blocks")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/329k [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (7275 > 512). Running this sequence through the model will result in indexing errors


README.md:   0%|          | 0.00/44.3k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

yor_Latn 5200 blocks
ibo_Latn 1200 blocks
hau_Latn 1200 blocks
swh_Latn 1200 blocks
eng_Latn 1200 blocks


In [ ]:
def as_ds(blocks):
    return Dataset.from_dict({"input_ids": blocks,
                              "attention_mask": [[1] * len(b) for b in blocks]})

# held-out eval sets: the LAST EVAL_BLOCKS of each language (never in either training set)
eval_sets = {l: as_ds(raw[l][-EVAL_BLOCKS:]) for l in LANGS}

# LAFT: all budget on Yoruba
laft_blocks = raw[TARGET][:N_BLOCKS]
# MAFT: equal share per language; Yoruba share = first per_lang blocks of the LAFT set
maft_blocks = []
for l in LANGS:
    maft_blocks += raw[l][:per_lang]
random.Random(SEED).shuffle(maft_blocks)

train_laft, train_maft = as_ds(laft_blocks), as_ds(maft_blocks)
print("LAFT blocks:", len(train_laft), "| MAFT blocks:", len(train_maft))
print("Tokens per run ~", len(train_laft) * BLOCK / 1e6, "M")

LAFT blocks: 5000 | MAFT blocks: 5000
Tokens per run ~ 1.28 M


## 2. Adapt XLM-R (the paper's settings: lr 5e-5, batch 10, 15% masking)

In [ ]:
def adapt(train_ds, out_dir):
    set_seed(SEED)
    model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL)
    coll = DataCollatorForLanguageModeling(tok, mlm=True, mlm_probability=MLM_PROB)
    args = TrainingArguments(output_dir=out_dir, per_device_train_batch_size=BATCH,
                             learning_rate=LR, num_train_epochs=EPOCHS,
                             fp16=torch.cuda.is_available(), save_strategy="no",
                             logging_steps=200, report_to="none", seed=SEED,
                             dataloader_num_workers=2)
    tr = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=coll)
    tr.train()
    tr.save_model(out_dir); tok.save_pretrained(out_dir)
    del tr, model; gc.collect(); torch.cuda.empty_cache()

if not os.path.exists(f"{LAFT_DIR}/config.json"):
    adapt(train_laft, LAFT_DIR)
if not os.path.exists(f"{MAFT_DIR}/config.json"):
    adapt(train_maft, MAFT_DIR)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
200,2.759362
400,2.247701


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
200,2.583576
400,2.394232


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 3. Held-out MLM loss on every language (shows gains on Yoruba and forgetting elsewhere)

In [4]:
import gc
gc.collect(); torch.cuda.empty_cache()

def mlm_loss(model_path):
    model = AutoModelForMaskedLM.from_pretrained(model_path)
    coll = DataCollatorForLanguageModeling(tok, mlm=True, mlm_probability=MLM_PROB)
    args = TrainingArguments(output_dir="tmp_eval", per_device_eval_batch_size=4,
                             fp16=torch.cuda.is_available(), report_to="none")
    tr = Trainer(model=model, args=args, data_collator=coll)
    res = {}
    for l in LANGS:
        set_seed(0)
        res[l] = tr.evaluate(eval_dataset=eval_sets[l])["eval_loss"]
    del tr, model; gc.collect(); torch.cuda.empty_cache()
    return res

mlm_rows = {}
for name, path in [("xlm-r", BASE_MODEL), ("mini-LAFT", LAFT_DIR), ("mini-MAFT", MAFT_DIR)]:
    mlm_rows[name] = mlm_loss(path)
mlm_df = pd.DataFrame(mlm_rows).T.round(3)
mlm_df

NameError: name 'torch' is not defined

In [ ]:
SYSTEMS = {"xlm-r": BASE_MODEL, "mini-LAFT": LAFT_DIR, "mini-MAFT": MAFT_DIR,
           "AfroXLMR (official)": OFFICIAL_AFRO}

for k, v in SYSTEMS.items():
    print(k, "->", v)

xlm-r -> xlm-roberta-base
mini-LAFT -> /content/drive/MyDrive/yoruba_project/mini_maft_run/mini_laft
mini-MAFT -> /content/drive/MyDrive/yoruba_project/mini_maft_run/mini_maft
AfroXLMR (official) -> Davlan/afro-xlmr-base


NameError: name 'finetune_eval' is not defined

## 4. SIB-200 evaluation
* `n = 0` (zero-shot transfer): fine-tune on the full **English** train split, test on each language.
  One fine-tune per (model, seed) evaluates all 5 languages.
* `n > 0`: fine-tune on `n` Yoruba examples per class only, test on Yoruba.

In [ ]:
sib = {l: load_dataset("Davlan/sib200", l) for l in LANGS}      # CHECK: loads without a script error
label_col = "category" if "category" in sib["eng_Latn"]["train"].column_names else "label"
classes = sorted(set(sib["eng_Latn"]["train"][label_col]))
lab2id = {c: i for i, c in enumerate(classes)}
print("classes:", classes)

def encode(split_ds, tk):
    enc = tk(split_ds["text"], truncation=True, max_length=CLS_MAX_LEN)
    enc["labels"] = [lab2id[x] for x in split_ds[label_col]]
    return Dataset.from_dict(dict(enc))

def few_shot(split_ds, n, seed):
    rng = random.Random(seed)
    by = {}
    for i, c in enumerate(split_ds[label_col]):
        by.setdefault(c, []).append(i)
    idx = [j for c in classes for j in rng.sample(by[c], min(n, len(by[c])))]
    return split_ds.select(idx)

def metrics(p):
    pred = p.predictions.argmax(-1)
    return {"acc": accuracy_score(p.label_ids, pred),
            "f1": f1_score(p.label_ids, pred, average="macro")}

def finetune_eval(model_path, train_split, test_langs, seed):
    tk = AutoTokenizer.from_pretrained(model_path)
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=len(classes))
    tr_ds = encode(train_split, tk)
    epochs = 30 if len(tr_ds) <= 70 else 15 if len(tr_ds) <= 350 else 10
    args = TrainingArguments(output_dir="tmp_cls", per_device_train_batch_size=16,
                             per_device_eval_batch_size=64, learning_rate=2e-5,
                             num_train_epochs=epochs, fp16=torch.cuda.is_available(),
                             save_strategy="no", report_to="none", seed=seed, logging_steps=1000)
    tr = Trainer(model=model, args=args, train_dataset=tr_ds,
                 data_collator=DataCollatorWithPadding(tk), compute_metrics=metrics)
    tr.train()
    out = {}
    for l in test_langs:
        out[l] = tr.evaluate(eval_dataset=encode(sib[l]["test"], tk))
    del tr, model; gc.collect(); torch.cuda.empty_cache()
    return {l: {"acc": v["eval_acc"], "f1": v["eval_f1"]} for l, v in out.items()}

README.md:   0%|          | 0.00/47.9k [00:00<?, ?B/s]

train.tsv:   0%|          | 0.00/128k [00:00<?, ?B/s]

dev.tsv:   0%|          | 0.00/17.0k [00:00<?, ?B/s]

test.tsv:   0%|          | 0.00/37.6k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

... (22 outputs left out)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

classes: ['entertainment', 'geography', 'health', 'politics', 'science/technology', 'sports', 'travel']


In [ ]:

# --- pilot settings (override the config cell) ---
SEEDS = [1]
SHOTS = [5, 25, 100]

# --- local copies of the adapted models: much faster to load than Drive ---
import shutil, time
for src, dst in [(LAFT_DIR, "/content/mini_laft"), (MAFT_DIR, "/content/mini_maft")]:
    if not os.path.exists(dst):
        shutil.copytree(src, dst)

SYSTEMS = {"xlm-r": BASE_MODEL,
           "mini-LAFT": "/content/mini_laft",
           "mini-MAFT": "/content/mini_maft",
           "AfroXLMR (official)": OFFICIAL_AFRO}
print(SYSTEMS)

# --- resume-safe grid ---
RES_PATH = f"{WORK}/sib200_results.csv"
rows = pd.read_csv(RES_PATH).to_dict("records") if os.path.exists(RES_PATH) else []
done = {(r["system"], r["lang"], r["n"], r["seed"]) for r in rows}

def save():
    pd.DataFrame(rows).to_csv(RES_PATH, index=False)

for sysname, path in SYSTEMS.items():
    for seed in SEEDS:
        t0 = time.time()
        # zero-shot: fine-tune on English, evaluate on all 5 languages
        if any((sysname, l, 0, seed) not in done for l in LANGS):
            r = finetune_eval(path, sib["eng_Latn"]["train"], LANGS, seed)
            for l, m in r.items():
                rows.append(dict(system=sysname, lang=l, n=0, seed=seed, **m))
            save()
        # Yoruba few-shot curve
        for n in SHOTS:
            if (sysname, TARGET, n, seed) in done:
                continue
            sub = few_shot(sib[TARGET]["train"], n, seed)
            r = finetune_eval(path, sub, [TARGET], seed)
            rows.append(dict(system=sysname, lang=TARGET, n=n, seed=seed, **r[TARGET]))
            save()
        print(f"done {sysname} seed {seed}: {(time.time() - t0) / 60:.1f} min")

res = pd.DataFrame(rows)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


Training Loss,Validation Loss,Step,Acc,F1
No log,3.528257,440,0.343137,0.254120


Training Loss,Validation Loss,Step,Acc,F1
No log,3.479644,440,0.333333,0.235168


Training Loss,Validation Loss,Step,Acc,F1
No log,2.213298,440,0.578431,0.556209


... (394 outputs left out)

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


Training Loss,Validation Loss,Step,Acc,F1
No log,1.164829,370,0.632353,0.620841


done AfroXLMR (official) seed 3


## 5. Results

In [ ]:
yo = res[res.lang == TARGET].groupby(["system", "n"]).f1.agg(["mean", "std"]).round(3)
print(yo.unstack(0)["mean"])

zs = res[res.n == 0].groupby(["system", "lang"]).f1.mean().unstack().round(3)
print("\nZero-shot transfer from English (macro-F1):")
zs

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 4))
for s in SYSTEMS:
    d = res[(res.system == s) & (res.lang == TARGET)].groupby("n").f1.agg(["mean", "std"])
    ax.errorbar(d.index, d["mean"], yerr=d["std"], marker="o", capsize=3, label=s)
ax.set_xscale("symlog", linthresh=1); ax.set_xlabel("labelled Yoruba examples per class (0 = English transfer)")
ax.set_ylabel("macro-F1 on SIB-200 Yoruba test"); ax.legend(); fig.tight_layout()
fig.savefig("yoruba_data_efficiency.png", dpi=160)

## Interpreting this honestly
* SIB-200 test has only ~204 examples per language, so differences of a couple of F1 points are noise.
  Use the std across seeds, and report that.
* The three adapted systems use *far* less text than the real AfroXLMR, so mini-MAFT/LAFT are not
  expected to match it. The meaningful comparison is **mini-LAFT vs mini-MAFT**.
* Hausa and Swahili were already in XLM-R's pretraining, Yoruba and Igbo were not; check whether the
  gain from adaptation differs between those groups in the zero-shot table.